# Imports

In [ ]:
import pandas as pd

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split

import serialTools
import captureTools
import eval

In [ ]:
datasetsPath = '../datasets/harus/'

In [ ]:
with open(datasetsPath + 'UCI HAR Dataset/features.txt') as f:
    xNames = [] # List of column/feature names

    for line in f:                          # Reading each line
        parts = line.strip().split(' ')     # Splitting each line by space

        if len(parts) > 1:                  # If the line has more than 1 element
            label = parts[1]                # The second element is the label
            dupe = False                    # Label is not a dupe, yet!
            index = 0                       # Rising index for naming
            amount = xNames.count(label)    # Check, if the label is already in the list
            
            while amount!=0:                # If a label is already in the list, add a number (index) to it 
                index = index+1
                newName = label + '_' + str(index)
                amount = xNames.count(newName)
                dupe = True
            if dupe == True:                # If the label has been a duplicate, use the new name
                xNames.append(newName)    
            else:
                xNames.append(label)        # Otherwise, just use the original name

In [ ]:
xtrain = pd.DataFrame(columns=xNames)         # Create dataframes with columns named after features.txt
xtest = pd.DataFrame(columns=xNames)

with open(datasetsPath + 'UCI HAR Dataset/train/X_train.txt', 'r') as f:
    for line in f:
        liste = line.strip().split(' ')      # Create a list of every object in the list thats seperated by " "
        liste = [i for i in liste if i != ''] # Remove empty strings
        liste = [float(i) for i in liste]     # Cast every object in the list to float
        xtrain.loc[len(xtrain)] = liste          # Add new_list as a new row to the dataframe

with open(datasetsPath + 'UCI HAR Dataset/test/X_test.txt', 'r') as f:
    for line in f:
        liste = line.strip().split(' ')      # Create a list of every object in the list thats seperated by " "
        liste = [i for i in liste if i != ''] # Remove empty strings
        liste = [float(i) for i in liste]     # Cast every object in the list to float
        xtest.loc[len(xtest)] = liste          # Add new_list as a new row to the dataframe

In [ ]:
ytrain = pd.DataFrame(columns=['label'])    # Create dataframes for feature labels
ytest = pd.DataFrame(columns=['label'])

with open(datasetsPath + 'UCI HAR Dataset/train/y_train.txt', 'r') as f:
    labels = []
    for line in f:
        labels.append(int(line.strip())-1)
    
ytrain['label'] = labels

with open(datasetsPath + 'UCI HAR Dataset/test/y_test.txt', 'r') as f:
    labels = []
    for line in f:
        labels.append(int(line.strip())-1)
    
ytest['label'] = labels

# Pre-Processing

In [ ]:
for col in xtrain.columns:
    xtrain[col] = xtrain[col].astype('float32')

for col in ytrain.columns:
    ytrain[col] = ytrain[col].astype('int32')

evaldata=[(xtrain,ytrain),(xtest,ytest)]          # Datensatz zur Evaluierung

## Global Variables

In [ ]:
donor =     XGBClassifier()
final =     XGBClassifier()
bestIter =  0
comport =   '/dev/ttyACM0'
baudrate =  9600

## Functions

In [ ]:
def traindonor(model):
    global bestIter

    model.set_params(
        objective='multi:softmax',          # Multi-Klassifizierung
        num_class=6,
        learning_rate=0.1,
        n_estimators=1000,                  # "Große Anzahl an Schaetzern, die nicht erreicht werden soll"
        early_stopping_rounds=50,           # Anzahl an Runden, bei denen sich das Modell nicht verbessern muss, bis abgebrochen wird
        max_depth=3
    )

    model.fit(
        xtrain, ytrain, 
        eval_set=evaldata, 
        verbose=False
    )

    bestIter = model.best_iteration

def trainfinal(model):
    model.set_param(
        objective='multi:softmax',      
        num_class=6,                    
        learning_rate=0.1,              
        n_estimators=bIter,                   
        num_parallel_tree=1,            # m2c workaround
        max_depth=3,                    
    )
    
    model.fit(
        xtrain, ytrain, 
        eval_set=evaldata, 
        verbose=False
    )




